In [1]:
import numpy as np
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pandas as pd
import os
import sys
sys.path.append("D:\\Program Files\\Lumerical\\v241\\api\\python\\")
import lumapi

In [2]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

In [3]:
class PWBParameters:
    def __init__(self):
        self.L = 250e-6
        self.h = 150e-6
        self.r1 = 6.2e-6
        self.r = 0.8e-6
        self.R = 60e-6
        self.r2 = 7e-6
        self.l1 = 100e-6
        # self.l2 = 60e-6
        self.curve_points = 100
        self.wavelength = 1.55e-6
        self.mesh_accuracy = 1 
        self.simulation_time = 2000e-15 

In [30]:
def generate_pwb_path_1(params):
    curve_points = params.curve_points
    L = params.L
    h = params.h
    r1 = params.r1
    r = params.r
    R = params.R
    r2 = params.r2
    l1 = params.l1
    l2 = h - R
    l = L - l1 - R

    # taper1
    x1 = np.linspace(0, l1, curve_points)
    y1 = np.zeros(curve_points)
    z1 = np.zeros(curve_points) 

    # bending
    angle_start = 0
    angle_end = np.pi / 2
    t = np.linspace(angle_start, angle_end, 2*curve_points)
    center_x = l1 + l
    center_z = -R
    x2 = center_x + R * np.sin(t)
    y2 = np.zeros_like(x2)
    z2 = center_z + R * np.cos(t)

    # taper2
    z3 = np.linspace(z2[-1], z2[-1] - l2, curve_points)
    y3 = np.zeros(curve_points)
    x3 = np.zeros(curve_points) + x2[-1]

    x = np.concatenate((x1, x2, x3))
    y = np.zeros_like(x)
    z = np.concatenate((z1, z2, z3))
    path = np.column_stack((x, y, z))
    return path

In [4]:
def generate_pwb_path_2(params):
    curve_points = params.curve_points
    L = params.L
    h = params.h
    r1 = params.r1
    r = params.r
    R = params.R
    r2 = params.r2
    l1 = params.l1
    l2 = h - R
    l = L - l1 - R

    # taper1
    x1 = np.linspace(0, l1, curve_points)
    y1 = np.zeros(curve_points)
    z1 = np.zeros(curve_points) 

    # bending
    angle_start = 0
    angle_end = np.pi / 2
    t = np.linspace(angle_start, angle_end, 2*curve_points)
    center_x = l1 + l
    center_z = -R
    x2 = center_x + R * np.sin(t)
    y2 = np.zeros_like(x2)
    z2 = center_z + R * np.cos(t)

    # taper2
    z3 = np.linspace(z2[-1], z2[-1] - l2, curve_points)
    y3 = np.zeros(curve_points)
    x3 = np.zeros(curve_points) + x2[-1]

    x = np.concatenate((x1, x2, x3))
    y = np.zeros_like(x)
    z = np.concatenate((z1, z2, z3))
    path = np.column_stack((x, y, z))
    return path

In [37]:
def generate_pwb_structure_1(fdtd, params):
    """按照参数生成对应结构"""
    L = params.L
    h = params.h
    r1 = params.r1
    r = params.r
    r2 = params.r2
    l1 = params.l1
    R = params.R
    l2 = h - R
    l = L - l1 - R
    path = generate_pwb_path_1(params)

    fdtd.deleteall()
    fdtd.load("D:/simulation/Simulation Project/simulation/PD-PWB-SMF/SMF.fsp")
    fdtd.importmaterialdb("D:/simulation/Simulation Project/simulation/database.mdf")

    # 创建taper1
    # num_segments_1 = 100
    # path_length_1 = 100
    # segment_length_1 = path_length_1 // num_segments_1
    # for i in range(num_segments_1):
    #     start_idx = i * segment_length_1
    #     end_idx = min((i + 1) * segment_length_1, path_length_1)

    #     if start_idx >= end_idx:
    #         continue

    #     # 获取段的起始和结束位置
    #     start_pos = path[start_idx]
    #     end_pos = path[end_idx]

    #     # 计算段的中心位置
    #     center_x = (start_pos[0] + end_pos[0]) / 2
    #     center_y = (start_pos[1] + end_pos[1]) / 2
    #     center_z = (start_pos[2] + end_pos[2]) / 2

    #     # 计算段的长度
    #     segment_len = np.linalg.norm(end_pos - start_pos)

    #     radius_1 = r1 - (r1 - r)*i / 100

    #     # 计算方向向量
    #     direction = end_pos - start_pos
    #     norm = np.linalg.norm(direction)
    #     if norm > 0:
    #         direction = direction / norm
    #     else:
    #         direction = np.array([1, 0, 0])  
    #     dz, dy, dx = direction[2], direction[1], direction[0]
    #     theta = np.arctan2(dx, dz) * 180 / np.pi
    #     phi = np.arcsin(dy) * 180 / np.pi
    #     fdtd.addcircle()
    #     fdtd.set("name", f"taper1_{i}")
    #     fdtd.set("material", "Vancore B")
    #     fdtd.set("make ellipsoid", 0)
    #     fdtd.set("x", center_x)
    #     fdtd.set("y", center_y)
    #     fdtd.set("z", center_z)
    #     fdtd.set("radius", radius_1)
    #     fdtd.set("z span", segment_len)
    #     fdtd.set("first axis", "y")
    #     fdtd.set("rotation 1", theta)
    #     fdtd.set("second axis", "x")
    #     fdtd.set("rotation 2", phi)

    # 创建taper1
    ### 问题是每次都需要重新点击apply或ok才能正常
    # fdtd.addcustom()
    # fdtd.set("name", "Taper 1")
    # fdtd.set("material","Vancore B")
    # fdtd.set("x", l1 / 2)        
    # fdtd.set("y", 0)
    # fdtd.set("z", 0)
    # fdtd.set("create 3D object by", "revolution")                   
    # fdtd.slope = (r - r1) / l1       
    # fdtd.offset = ((r1 + r) / 2 )*1e6
    # fdtd.eqn = f"{fdtd.slope}*x + {fdtd.offset}"
    # # fdtd.offset = r1 + fdtd.slope * (l1 / 2)
    # # fdtd.eqn = fdtd.num2str(fdtd.slope*1e6) + "*x*1e-6 + " + fdtd.num2str(fdtd.offset*1e6) 
    # fdtd.set("equation 1", fdtd.eqn)  
    # fdtd.set("x span", l1)                     
    # fdtd.set("y span", 2 * r1)                    
    # fdtd.set("z span", 2 * r1)        
    
    # 创建straight
    # fdtd.addcircle()
    # fdtd.set("name", "Straight")
    # fdtd.set("material", "Vancore B")
    # fdtd.set("x", l1 + l / 2)
    # fdtd.set("y", 0)
    # fdtd.set("z", 0)
    # fdtd.set("make ellipsoid", 0)
    # fdtd.set("radius", r)
    # fdtd.set("z span", l)
    # fdtd.set("first axis", "y")
    # fdtd.set("rotation 1", 90)

    # 创建bending
    num_segments_2 = 200
    path_length_2 = 200
    segment_length_2 = path_length_2 // num_segments_2
    for i in range(num_segments_2):
        start_idx = i * segment_length_2 + 100
        end_idx = min((i + 1) * segment_length_2, path_length_2) + 100

        if start_idx >= end_idx:
            continue

        # 获取段的起始和结束位置
        start_pos = path[start_idx]
        end_pos = path[end_idx]

        # 计算段的中心位置
        center_x = (start_pos[0] + end_pos[0]) / 2
        center_y = (start_pos[1] + end_pos[1]) / 2
        center_z = (start_pos[2] + end_pos[2]) / 2

        # 计算段的长度
        segment_len = np.linalg.norm(end_pos - start_pos)
       
        # 计算方向向量
        direction = end_pos - start_pos
        norm = np.linalg.norm(direction)
        if norm > 0:
            direction = direction / norm
        else:
            direction = np.array([1, 0, 0])  
        dz, dy, dx = direction[2], direction[1], direction[0]
        theta = np.arctan2(dx, dz) * 180 / np.pi  # 绕y轴
        phi = np.arcsin(dy) * 180 / np.pi         # 绕x轴
        fdtd.addcircle()
        fdtd.set("name", f"Bending_{i}")
        fdtd.set("material", "Vancore B")
        fdtd.set("make ellipsoid", 0)
        fdtd.set("x", center_x)
        fdtd.set("y", center_y)
        fdtd.set("z", center_z)
        fdtd.set("radius", r)
        fdtd.set("z span", segment_len)
        fdtd.set("first axis", "y")
        fdtd.set("rotation 1", theta)
        fdtd.set("second axis", "x")
        fdtd.set("rotation 2", phi)

In [5]:
def generate_pwb_structure_2(fdtd, params):
    """按照参数生成对应结构"""
    L = params.L
    h = params.h
    r1 = params.r1
    r = params.r
    r2 = params.r2
    l1 = params.l1
    R = params.R
    l2 = h - R
    l = L - l1 - R
    path = generate_pwb_path_2(params)

    fdtd.deleteall()
    fdtd.load("D:/simulation/Simulation Project/simulation/PD-PWB-SMF/SMF.fsp")
    fdtd.importmaterialdb("D:/simulation/Simulation Project/simulation/database.mdf")
    
    # 创建bending
    num_segments_2 = 200
    path_length_2 = 200
    segment_length_2 = path_length_2 // num_segments_2
    for i in range(num_segments_2):
        start_idx = i * segment_length_2 + 100
        end_idx = min((i + 1) * segment_length_2, path_length_2) + 100

        if start_idx >= end_idx:
            continue

        # 获取段的起始和结束位置
        start_pos = path[start_idx]
        end_pos = path[end_idx]

        # 计算段的中心位置
        center_x = (start_pos[0] + end_pos[0]) / 2
        center_y = (start_pos[1] + end_pos[1]) / 2
        center_z = (start_pos[2] + end_pos[2]) / 2

        # 计算段的长度
        segment_len = np.linalg.norm(end_pos - start_pos)
       
        # 计算方向向量
        direction = end_pos - start_pos
        norm = np.linalg.norm(direction)
        if norm > 0:
            direction = direction / norm
        else:
            direction = np.array([1, 0, 0])  
        dz, dy, dx = direction[2], direction[1], direction[0]
        theta = np.arctan2(dx, dz) * 180 / np.pi  # 绕y轴
        phi = np.arcsin(dy) * 180 / np.pi         # 绕x轴
        fdtd.addcircle()
        fdtd.set("name", f"Bending_{i}")
        fdtd.set("material", "Vancore B")
        fdtd.set("make ellipsoid", 0)
        fdtd.set("x", center_x)
        fdtd.set("y", center_y)
        fdtd.set("z", center_z)
        fdtd.set("radius", r)
        fdtd.set("z span", segment_len)
        fdtd.set("first axis", "y")
        fdtd.set("rotation 1", theta)
        fdtd.set("second axis", "x")
        fdtd.set("rotation 2", phi)

    # 创建taper2
    num_segments_3 = 100
    path_length_3 = 100
    segment_length_3 = path_length_3 // num_segments_3
    for i in range(num_segments_3):
        radius_2 = r + (r2 - r)*(i + 1) / 100
        fdtd.addcircle()
        fdtd.set("name", f"taper2_{i}")
        fdtd.set("material", "Vancore B")
        fdtd.set("make ellipsoid", 0)
        fdtd.set("x", L)
        fdtd.set("y", 0)
        fdtd.set("z", -R - (i + 1/2) * l2 / 100)
        fdtd.set("radius", radius_2)
        fdtd.set("z span", l2 / 100)

In [38]:
def setup_fdtd_simulation_1(fdtd, params):
    """设置仿真参数"""
    L = params.L
    h = params.h
    r1 = params.r1
    r = params.r
    R = params.R
    r2 = params.r2
    l1 = params.l1
    l2 = h - R
    l = L - l1 - R

    x_min, x_max = L - R, L
    y_min, y_max = 0, 0
    z_min, z_max = -R, 0
    margin = 6e-6

    fdtd.setresource("FDTD","GPU", True)
    fdtd.addfdtd()
    fdtd.set("dimension", "3D")
    fdtd.set("x min", x_min)
    fdtd.set("x max", x_max + margin)
    fdtd.set("y min", y_min - margin)
    fdtd.set("y max", y_max + margin)
    fdtd.set("z min", z_min)
    fdtd.set("z max", z_max + margin)
    fdtd.set("express mode", True)
    fdtd.set("x min bc", "PML")
    fdtd.set("x max bc", "PML")
    fdtd.set("y min bc", "PML")
    fdtd.set("y max bc", "PML")
    fdtd.set("z min bc", "PML")
    fdtd.set("z max bc", "PML")
    fdtd.set("mesh accuracy", params.mesh_accuracy)
    fdtd.set("mesh type", "auto non-uniform")
    fdtd.set("simulation time", params.simulation_time)
    
    fdtd.addmode()
    fdtd.set("name", "source")
    fdtd.set("injection axis", "x-axis")
    fdtd.set("direction", "forward")
    fdtd.set("x", x_min + 1e-6)
    fdtd.set("y", 0)
    fdtd.set("z", 0)
    fdtd.set("y span", 2 * margin)
    fdtd.set("z span", 2 * margin)
    fdtd.set("wavelength start", params.wavelength)
    fdtd.set("wavelength stop", params.wavelength)

    fdtd.addpower()
    fdtd.set("name", "monitor_1")
    fdtd.set("monitor type", "2D Z-normal")
    fdtd.set("x", L)
    fdtd.set("y", 0)
    fdtd.set("z", -R)
    fdtd.set("y span", 2 * margin)
    fdtd.set("x span", 2 * margin)

    fdtd.addpower()
    fdtd.set("name", "transmission_monitor")
    fdtd.set("monitor type", "2D Y-normal")
    fdtd.set("y", 0)
    fdtd.set("x", (2 * L  - R)/ 2)
    fdtd.set("x span", R + 6e-6)
    fdtd.set("z", -R / 2)
    fdtd.set("z span", R + 6e-6)

In [6]:
def setup_fdtd_simulation_2(fdtd, params):
    """设置仿真参数"""
    L = params.L
    h = params.h
    r1 = params.r1
    r = params.r
    R = params.R
    r2 = params.r2
    l1 = params.l1
    l2 = h - R
    l = L - l1 - R

    x_min, x_max = L - R, L
    y_min, y_max = 0, 0
    z_min, z_max = -h, 0
    margin = 10e-6

    fdtd.setresource("FDTD","GPU", True)
    fdtd.addfdtd()
    fdtd.set("dimension", "3D")
    fdtd.set("x min", x_min - margin)
    fdtd.set("x max", x_max + margin)
    fdtd.set("y min", y_min - margin)
    fdtd.set("y max", y_max + margin)
    fdtd.set("z min", z_min - margin)
    fdtd.set("z max", z_max + margin)
    fdtd.set("express mode", True)
    fdtd.set("x min bc", "PML")
    fdtd.set("x max bc", "PML")
    fdtd.set("y min bc", "PML")
    fdtd.set("y max bc", "PML")
    fdtd.set("z min bc", "PML")
    fdtd.set("z max bc", "PML")
    fdtd.set("mesh accuracy", params.mesh_accuracy)
    fdtd.set("mesh type", "auto non-uniform")
    fdtd.set("simulation time", params.simulation_time)
    
    fdtd.addmode()
    fdtd.set("name", "source")
    fdtd.set("injection axis", "x-axis")
    fdtd.set("direction", "forward")
    fdtd.set("x", L - R + 2e-6) 
    fdtd.set("y", 0)
    fdtd.set("z", z_max)
    fdtd.set("y span", 3e-6)
    fdtd.set("z span", 3e-6)
    fdtd.set("wavelength start", params.wavelength)
    fdtd.set("wavelength stop", params.wavelength)

    fdtd.addpower()
    fdtd.set("name", "monitor_1")
    fdtd.set("monitor type", "2D Z-normal")
    fdtd.set("x", L)
    fdtd.set("y", 0)
    fdtd.set("z", -h)
    fdtd.set("y span", 2 * margin)
    fdtd.set("x span", 2 * margin)

    fdtd.addpower()
    fdtd.set("name", "transmission_monitor")
    fdtd.set("monitor type", "2D Y-normal")
    fdtd.set("y", 0)
    fdtd.set("x", (2 * L - R) / 2)
    fdtd.set("x span", R + 2 * margin)
    fdtd.set("z", -h/ 2)
    fdtd.set("z span", h + 2 * margin)

In [33]:
def get_data_1(fdtd, params):
    """运行仿真并分析结果"""

    source_E = fdtd.getresult("source", "mode profile")
    transmission_E = fdtd.getresult("transmission_monitor", "E")
    transmission_P = fdtd.getresult("transmission_monitor", "P")
    monitor_1_E = fdtd.getresult("monitor_1", "E")
    T_total = fdtd.getresult("monitor_1", "T")
    
    results = {
        'source_E': source_E,
        'monitor_1_E': monitor_1_E,
        'transmission_E': transmission_E,
        'transmission_P': transmission_P,
        'T_total': T_total["T"][0]
    }
    
    return results

In [7]:
def get_data_2(fdtd, params):
    """运行仿真并分析结果"""

    source_E = fdtd.getresult("source", "mode profile")
    transmission_E = fdtd.getresult("transmission_monitor", "E")
    transmission_P = fdtd.getresult("transmission_monitor", "P")
    monitor_1_E = fdtd.getresult("monitor_1", "E")
    T_total = fdtd.getresult("monitor_1", "T")
    
    results = {
        'source_E': source_E,
        'monitor_1_E': monitor_1_E,
        'transmission_E': transmission_E,
        'transmission_P': transmission_P,
        'T_total': T_total["T"][0]
    }
    
    return results

In [34]:
def visualize_and_save_results_1(fdtd, params):
    """可视化结果并保存T_total数据"""
    
    # 获取数据
    results = get_data_1(fdtd, params)
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    fig, ax = plt.subplots(figsize=(6, 5))
    trans_E = results['transmission_E']
    E_field = trans_E['E']
    Ex = E_field[:, 0, :, 0, 0]
    Ey = E_field[:, 0, :, 0, 1]
    Ez = E_field[:, 0, :, 0, 2]
    E_intensity_trans = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
    x_trans = np.squeeze(trans_E['x']) * 1e6  # (381,)
    z_trans = np.squeeze(trans_E['z']) * 1e6  # (101,)
    X_trans, Z_trans = np.meshgrid(x_trans, z_trans, indexing='ij')
    
    vmin = 1e-5
    vmax = np.nanmax(E_intensity_trans)
    data_to_plot = E_intensity_trans

    norm = LogNorm(vmin=vmin, vmax=vmax)
    pcm = ax.pcolormesh(X_trans, Z_trans, data_to_plot, cmap="inferno", shading='auto', norm=norm)

    # 对数色条
    cbar = fig.colorbar(pcm, ax=ax, pad=0.02, format=ticker.LogFormatterSciNotation())

    ax.set_xlabel('X (μm)')
    ax.set_ylabel('Z (μm)')
    ax.set_title('Transmission Monitor')

    plt.tight_layout()
    plt.savefig("D:/simulation/Simulation Project/simulation/PD-PWB-SMF/results/Pictures/Section1.png", dpi=300, bbox_inches='tight')
    plt.show()

    T_total = results['T_total']
    return T_total

In [40]:
params = PWBParameters()
fdtd = lumapi.FDTD()
generate_pwb_structure_1(fdtd, params)
setup_fdtd_simulation_1(fdtd, params)
fdtd.save("D:/simulation/Simulation Project/simulation/PD-PWB-SMF/test/test.fsp")

In [9]:
params = PWBParameters()
fdtd = lumapi.FDTD()
generate_pwb_structure_2(fdtd, params)

In [ ]:
# 单次运行
fdtd.run()
get_data_1(fdtd, params)
T_total = visualize_and_save_results_1(fdtd, params)
print(T_total)

In [50]:
fdtd.close()

In [ ]:
# r和R参数化扫描
results_dir = "D:/simulation/Simulation Project/simulation/PD-PWB-SMF/results/r_R_scan"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

# 定义参数范围
r_values = np.arange(0.4, 2.4, 0.2) * 1e-6
R_values = np.arange(20, 120, 10) * 1e-6
all_results = []

# 开始参数扫描
for i, r in enumerate(r_values):
    for j, R in enumerate(R_values):
        print(f"Processing r={r*1e6:.2f}μm, R={R*1e6:.1f}μm ({i*len(R_values)+j+1}/{len(R_values)*len(r_values)})")

        sys.path.append("D:\\Program Files\\Lumerical\\v241\\api\\python\\")
        import lumapi
        fdtd = lumapi.FDTD()
        
        # 更新参数
        params = PWBParameters()
        params.r = r
        params.R = R
        path = generate_pwb_path_1(params)
        generate_pwb_structure_1(fdtd, params)
        setup_fdtd_simulation_1(fdtd, params)
        fdtd.save("D:/simulation/Simulation Project/simulation/PD-PWB-SMF/test/test.fsp")
        fdtd.run()
        
        # 获取结果
        results = get_data_1(fdtd, params)
        T_total = results['T_total']
        
        # 保存结果到列表
        all_results.append({
            'r': r*1e6,
            'R': R*1e6,
            'T_total': T_total
        })
        
        # 生成文件名
        filename_base = f"r_{r*1e6:.2f}_R_{R*1e6:.1f}"
        
        # 保存图片
        try:
            plt.rcParams['font.family'] = 'Times New Roman'
            plt.rcParams['xtick.labelsize'] = 12
            plt.rcParams['ytick.labelsize'] = 12
            fig, ax = plt.subplots(figsize=(6, 5))
            trans_E = results['transmission_E']
            E_field = trans_E['E']
            Ex = E_field[:, 0, :, 0, 0]
            Ey = E_field[:, 0, :, 0, 1]
            Ez = E_field[:, 0, :, 0, 2]
            E_intensity_trans = np.abs(Ex)**2 + np.abs(Ey)**2 + np.abs(Ez)**2
            x_trans = np.squeeze(trans_E['x']) * 1e6 
            z_trans = np.squeeze(trans_E['z']) * 1e6  
            X_trans, Z_trans = np.meshgrid(x_trans, z_trans, indexing='ij')
            vmin = 1e-5
            vmax = np.nanmax(E_intensity_trans)
            data_to_plot = E_intensity_trans
            norm = LogNorm(vmin=vmin, vmax=vmax)
            pcm = ax.pcolormesh(X_trans, Z_trans, data_to_plot, cmap="inferno", shading='auto', norm=norm)
            cbar = fig.colorbar(pcm, ax=ax, pad=0.02, format=ticker.LogFormatterSciNotation())
            ax.set_xlabel('X (μm)')
            ax.set_ylabel('Z (μm)')
            ax.set_title('Transmission Monitor')
            plt.tight_layout()
            
            # 保存图片
            fig_path = os.path.join(results_dir, f"{filename_base}_field_plot.png")
            plt.savefig(fig_path, dpi=300, bbox_inches='tight')
            plt.close()
            
            print(f"  T_total = {T_total:.4f}, 图片已保存: {fig_path}")
            
        except Exception as e:
            print(f"  绘图时出错: {e}")
            print(f"  T_total = {T_total:.4f}")
        
        fdtd.close()

# 保存所有T_forward结果到文件
results_file = os.path.join(results_dir, "T_total_sweep_results.txt")
with open(results_file, 'w') as f:
    f.write("r(μm)\tR(μm)\tT_total\n")
    for result in all_results:
        f.write(f"{result['r']:.2f}\t{result['R']:.1f}\t{result['T_total']:.6f}\n")

In [ ]:
def plot_Ttotal_loss_heatmap_rR(data_file, output_dir=None, cmap='viridis_r', eps=1e-12, show=True):
    """
    读取 r, R, T_total 数据文件，计算 loss = -10*log10(| -T_total |) 并绘制热力图。
    - data_file: 带表头的 txt/tsv 文件（例如: r(μm)\\tR(μm)\\tT_total）
    - output_dir: 图片保存目录，None 则使用数据文件同目录
    - 返回 (loss_matrix, r_vals, R_vals)
    """
    # 读取文件（尝试若干编码）
    try:
        df = pd.read_csv(data_file, sep=r'\s*\t\s*', engine='python', encoding='utf-8')
    except Exception:
        try:
            df = pd.read_csv(data_file, sep=r'\s*\t\s*', engine='python', encoding='gbk')
        except Exception:
            df = pd.read_csv(data_file, sep=r'\s*\t\s*', engine='python', encoding='latin-1')

    # 简单清理列名
    cols = [c.strip() for c in df.columns]
    df.columns = cols

    # 若恰好3列，按顺序认定为 r, R, T_total
    if len(cols) == 3:
        r_col, R_col, T_col = cols[0], cols[1], cols[2]
    else:
        # 备选：按名字匹配
        lower = [c.lower() for c in cols]
        r_col = cols[lower.index(next(x for x in lower if x.startswith('r') ))] if any(x.startswith('r') for x in lower) else cols[0]
        R_col = cols[lower.index(next(x for x in lower if x.startswith('r') and x != r_col.lower()))] if len(cols)>1 else cols[1]
        T_col = next((c for c in cols if 't' in c.lower() or 'T' in c), cols[-1])

    # 提取数值
    r_vals = np.unique(df[r_col].values.astype(float))
    R_vals = np.unique(df[R_col].values.astype(float))

    r_vals = np.sort(r_vals)
    R_vals = np.sort(R_vals)

    # 构建矩阵
    loss_matrix = np.full((len(r_vals), len(R_vals)), np.nan)

    for _, row in df.iterrows():
        try:
            i = int(np.argwhere(r_vals == float(row[r_col]))[0])
            j = int(np.argwhere(R_vals == float(row[R_col]))[0])
            T = float(row[T_col])
            # 先取相反数使其为正，再取绝对值保护
            T_pos = -1.0 * T
            T_pos = np.abs(T_pos)
            T_pos = np.clip(T_pos, eps, None)
            loss = -10.0 * np.log10(T_pos)
            loss_matrix[i, j] = loss
        except Exception:
            # 如果匹配不上值（浮点精度问题），用 nearest index 填充
            ri = int(np.argmin(np.abs(r_vals - float(row[r_col])))
)
            rj = int(np.argmin(np.abs(R_vals - float(row[R_col])))
)
            T = float(row[T_col])
            T_pos = np.clip(np.abs(-1.0 * T), eps, None)
            loss_matrix[ri, rj] = -10.0 * np.log10(T_pos)

    # 绘图
    plt.rcParams['font.family'] = 'Times New Roman'
    fig, ax = plt.subplots(figsize=(8,6))
    im = ax.imshow(loss_matrix, origin='lower', aspect='auto', cmap=cmap)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label('loss (dB)', fontsize=12)

    # ticks 映射为真实值
    ax.set_xticks(np.arange(len(R_vals)))
    ax.set_xticklabels([f"{v:.0f}" if (v==int(v)) else f"{v:.1f}" for v in R_vals], rotation=45)
    ax.set_yticks(np.arange(len(r_vals)))
    ax.set_yticklabels([f"{v:.2f}" if v<10 else f"{v:.1f}" for v in r_vals])

    ax.set_xlabel('R (μm)', fontsize=12)
    ax.set_ylabel('r (μm)', fontsize=12)
    ax.set_title('Loss heatmap', fontsize=14)

    plt.tight_layout()

    if output_dir is None:
        output_dir = os.path.dirname(data_file) or '.'
    os.makedirs(output_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(data_file))[0]
    out_path = os.path.join(output_dir, f"{base}_loss_heatmap.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)

    print("Saved heatmap to:", out_path)
    return loss_matrix, r_vals, R_vals

In [ ]:
data_file = "D:/simulation/Simulation Project/simulation/PD-PWB-SMF/results/r_R_scan/T_total_sweep_results.txt"
loss_matrix, r_vals, R_vals = plot_Ttotal_loss_heatmap_rR(data_file, output_dir=None)

In [ ]:
# h和R参数化扫描,R(20,110),h(120,180)-->l2
results_dir = "D:/simulation/Simulation Project/simulation/PD-PWB-SMF/results/h_R_scan"
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

# 定义参数范围
R_values = np.arange(100, 120, 10) * 1e-6
h_values = np.arange(120, 180, 10) * 1e-6
all_results = []

# 开始参数扫描
for i, R in enumerate(R_values):
    for j, h in enumerate(h_values):
        print(f"Processing R={R*1e6:.1f}μm, h={h*1e6:.1f}μm ({i*len(h_values)+j+1}/{len(h_values)*len(R_values)})")

        sys.path.append("D:\\Program Files\\Lumerical\\v241\\api\\python\\")
        import lumapi
        fdtd = lumapi.FDTD()
        
        # 更新参数
        params = PWBParameters()
        params.h = h
        params.R = R
        path = generate_pwb_path_2(params)
        generate_pwb_structure_2(fdtd, params)
        setup_fdtd_simulation_2(fdtd, params)
        fdtd.save("D:/simulation/Simulation Project/simulation/PD-PWB-SMF/test/test.fsp")
        fdtd.run()
        
        # 获取结果
        results = get_data_2(fdtd, params)
        T_total = results['T_total']
        
        # 保存结果到列表
        all_results.append({
            'R': R*1e6,
            'h': h*1e6,
            'T_total': T_total
        })
        print(T_total)
        fdtd.close()

# 保存所有T_forward结果到文件
results_file = os.path.join(results_dir, "T_total_sweep_results.txt")
with open(results_file, 'w') as f:
    f.write("R(μm)\th(μm)\tT_total\n")
    for result in all_results:
        f.write(f"{result['R']:.1f}\t{result['h']:.1f}\t{result['T_total']:.6f}\n")